In [18]:
!pip install sentence-transformers faiss-cpu feedparser

In [17]:
import feedparser
def get_news():
  url="https://feeds.bbci.co.uk/news/rss.xml"
  feed = feedparser.parse(url)

  news_list=[]

  for entry in feed.entries[:15]:
    news_list.append(entry.title + " "+ entry.summary)

  return news_list
news = get_news()
for i,n in enumerate(news):
  print(i+1, n[:100])


1 Starmer sends 'chill' through civil service, union boss says The 'chill' follows the sacking of lead
2 I was left with an £8,000 vet bill when my insurer cancelled my pet policy Thousands of people have 
3 Why police are seeking to arrest billionaire K-pop mogul behind BTS Bang Si-hyuk, who created the su
4 How Leicester went from Premier League champions to League One in a decade Ten years ago, Leicester 
5 Ideal conditions to see peak of Lyrid meteor shower in UK The Lyrid meteor shower is the oldest reco
6 Bird flu vaccine trial against potential pandemic strain begins The jab targets the H5N1 flu strain 
7 Meta to track workers' clicks and keystrokes to train AI The firm will take data from the way employ
8 World's biggest maker of condoms set to raise prices due to Iran war Malaysia-based Karex produces m
9 O'Sullivan starts well in bid for eighth World Championship Ronnie O'Sullivan begins his quest for a
10 Foo Fighters interview: 'We're a different band without Taylor Hawkins

In [22]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
news_embeddings = model.encode(news)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [24]:
print(news_embeddings)

[[-0.05041727 -0.04334506 -0.01043671 ... -0.05968056 -0.03093004
   0.04814623]
 [ 0.01721686  0.07370553  0.06782072 ... -0.06034061 -0.02496201
   0.03951909]
 [-0.03799658  0.05037212 -0.01120516 ... -0.08009487  0.04937375
   0.02003018]
 ...
 [-0.01798676  0.00046033 -0.03965613 ... -0.06221984 -0.0354119
   0.02483462]
 [ 0.04352599  0.0479341   0.02237917 ... -0.05410905 -0.01715993
   0.00845397]
 [ 0.03339389 -0.03500116 -0.01170796 ... -0.05532031  0.06697016
   0.02773431]]


In [25]:
import faiss
import numpy as np

dimension = news_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(news_embeddings))

In [26]:
def ask_news_bot(question):
  q_embedding = model.encode([question])
  D,I = index.search(np.array(q_embedding),k=3)
  results = [news[i] for i in I[0]]

  return results

In [29]:
while True:
  q = input("\nAsk your question(or type exit): ")

  if q.lower() == "exit":
    break

  answers  =ask_news_bot(q)
  print("\nRelevant news:\n")
  for i, ans in enumerate(answers):
    print(f"{i+1}. {ans}\n")



Ask your question(or type exit): who is the president of india

Relevant news:

1. Henry Zeffman: Robbins's revelations are a dangerous moment for Starmer Drawing a line under Lord Mandelson's appointment is proving impossible for the prime minister.

2. The Papers: 'Starmer on the ropes' and 'Sobbin' Robbins spills the beans' The papers are dominated by the fallout for the prime minister from the failed vetting of Lord Mandelson for his role as British ambassador to the US.

3. Trump buys time for Iran deal after frantic day of diplomacy The US president's decision marked the second time in as many weeks he has backed off a threat to escalate the war, buying more time


Ask your question(or type exit):  what is happening in technology?

Relevant news:

1. Meta to track workers' clicks and keystrokes to train AI The firm will take data from the way employees work for its artificial intelligence models.

2. The Papers: 'Starmer on the ropes' and 'Sobbin' Robbins spills the beans' The p